# ROCm 10.2 on Radeon 780M (gfx1103)

This notebook runs every CPU/iGPU comparison provided by the project's benchmark scripts, using those scripts directly so workloads and reported metrics stay in sync. By default each comparison runs CPU first and then the ROCm iGPU; set `DEVICE_MODE` below to `"cpu"` or `"cuda"` to run only one side.

GPU jobs can hang on unstable stacks, so save your work before running the default comparison. Hugging Face model demos need internet access on first use. Notebook output is also saved to `benchmark_results/demo_notebook.txt`.

Local virtual environment: the ROCm runtime (`rocm-sdk-*`) and PyTorch are installed in `.venv`; system-wide `/opt/rocm` is not used. Select `.venv/bin/python` as the kernel, or run `uv run --with jupyterlab jupyter lab`.

In [1]:
import os
import platform
from pathlib import Path

import torch

from benchmark_logs import run_logged, start_notebook_logging, stop_notebook_logging

DEVICE_MODE = "both"  # Choose "both", "cpu", or "cuda".
if DEVICE_MODE not in ("both", "cpu", "cuda"):
    raise ValueError(f"Unsupported DEVICE_MODE: {DEVICE_MODE}")

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "benchmark_logs.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the dl-rocm-demo project")

notebook_log = start_notebook_logging()
gpu_available = torch.cuda.is_available()
if DEVICE_MODE == "cuda" and not gpu_available:
    stop_notebook_logging()
    raise RuntimeError("ROCm iGPU is unavailable")

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("HIP:", torch.version.hip)
print("iGPU available:", gpu_available)
print("HSA_OVERRIDE_GFX_VERSION:", os.environ.get("HSA_OVERRIDE_GFX_VERSION", "(not set)"))
if gpu_available:
    props = torch.cuda.get_device_properties(0)
    print("device:", torch.cuda.get_device_name(0))
    print("architecture:", props.gcnArchName)
    print(f"GTT memory: {props.total_memory / 2**30:.2f} GiB")

CHECK_GPU_SECTIONS = {
    "Test 1: Conv2D": "conv2d.txt",
    "Test 2: Transformer operations": "transformer.txt",
    "Test 3: Matmul": "matmul.txt",
    "Copying 256 MiB": "memory_copy.txt",
    "GPU memory": "gpu_memory.txt",
}


def run_benchmark(script, *args):
    print(f"\n===== {script} {' '.join(args)} =====")
    if script == "check_gpu.py":
        status = run_logged(
            PROJECT_ROOT / script,
            list(args),
            "gpu_environment.txt",
            CHECK_GPU_SECTIONS,
            full_file="check_gpu.txt",
        )
    else:
        log_file = {
            "train_demo.py": "training.txt",
            "computer_vision_demo.py": "computer_vision.txt",
            "transformers_demo.py": "transformers.txt",
            "sentence_transformers_demo.py": "sentence_transformers.txt",
            "resnet18_probe.py": "resnet18_probe.txt",
        }[script]
        status = run_logged(PROJECT_ROOT / script, list(args), log_file)
    if status:
        print(f"{script} exited with status {status}; see its saved log for details.")
    return status


python: 3.13.13
torch: 2.14.0+rocm10.2.0
HIP: 7.17.26434
iGPU available: True
HSA_OVERRIDE_GFX_VERSION: (not set)
device: AMD Radeon 780M Graphics
architecture: gfx1103
GTT memory: 7.44 GiB


## 1. PyTorch operation benchmarks

Runs the same comparisons as `check_gpu.py`: Conv2D, attention forward/backward, matrix multiplication, and 256 MiB memory-copy bandwidth. Each paired result reports CPU and iGPU measurements plus which device is faster; precision is shown for comparisons that use fp32 on CPU and fp16 on iGPU.

In [2]:
run_benchmark("check_gpu.py", "--device", DEVICE_MODE)


===== check_gpu.py --device both =====
ROCm / PyTorch check — local venv, no system-wide ROCm
python       : 3.13.13 (x86_64)
torch        : 2.14.0+rocm10.2.0
hip runtime  : 7.17.26434
cuda available: True
HSA_OVERRIDE : not set (correct)
device       : AMD Radeon 780M Graphics
arch         : gfx1103
memory (GTT) : 7.44 GiB
CU (reported): 6

Test 1: Conv2D forward, 16x3x128x128 -> 32 channels, 3x3
matmul sanity: 0.0250  (ok)


MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1103: libMIOpenCKGroupedConv_gfx1103.so: cannot open shared object file: No such file or directory


CPU fp32       10.78 ms        42.0 GFLOPS
iGPU fp16       0.76 ms       599.4 GFLOPS
CPU vs iGPU (Conv2D throughput (fp32 vs fp16)): CPU=42.022 GFLOPS, iGPU=599.351 GFLOPS; iGPU faster x14.26 (iGPU speedup x14.262666)

Test 2: Transformer operations — attention, Softmax, forward + backward
CPU fp32      224.25 ms        28.7 GFLOPS loss=0.00541 checks=ok
iGPU fp16       7.75 ms       831.7 GFLOPS loss=0.00532 checks=ok
CPU vs iGPU (attention throughput (fp32 vs fp16)): CPU=28.728 GFLOPS, iGPU=831.739 GFLOPS; iGPU faster x28.95 (iGPU speedup x28.951731)

Test 3: Matmul 4096x4096, FP16/BF16 (higher GFLOPS is better)
CPU fp32      203.71 ms       674.7 GFLOPS
CPU bf16       98.63 ms      1393.5 GFLOPS
iGPU fp16      21.63 ms      6355.2 GFLOPS
iGPU bf16      22.03 ms      6238.1 GFLOPS
CPU vs iGPU (matmul throughput (fp32 vs fp16)): CPU=674.683 GFLOPS, iGPU=6355.175 GFLOPS; iGPU faster x9.42 (iGPU speedup x9.4194954)

Copying 256 MiB (rough estimate of memory bandwidth)
CPU:    38.5 GB/s

0

## 2. Small CNN training

Runs the regular CPU/iGPU training comparison from `train_demo.py`, including loss, accuracy, memory use, and epoch-time speedup.

In [3]:
run_benchmark("train_demo.py", "--device", DEVICE_MODE)


===== train_demo.py --device both =====
train=4096 test=1024 epochs=2 batch=64 params=94,986
torch=2.14.0+rocm10.2.0 hip=7.17.26434



MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1103: libMIOpenCKGroupedConv_gfx1103.so: cannot open shared object file: No such file or directory


run            time   s/epoch     loss     acc  peak GiB
CPU fp32      1.91s     0.96s    0.005  100.0%      0.00
GPU fp16      0.82s     0.41s    0.005  100.0%      0.10
CPU vs iGPU (CNN epoch time): CPU=0.955 s/epoch, iGPU=0.411 s/epoch; iGPU faster x2.32 (iGPU speedup x2.3223385)


0

## 3. Computer vision training and inference

Runs both ResNet18 and MobileNetV3-Small comparisons from `computer_vision_demo.py`, including training-time and inference-latency speedups.

In [4]:
run_benchmark("computer_vision_demo.py", "--device", DEVICE_MODE)


===== computer_vision_demo.py --device both =====
Computer Vision training/inference | precision=fp32
images=64x64 train=128 test=64 batch=32 epochs=1
Dataset: synthetic class prototypes with noise; pretrained weights disabled.
ResNet18 CPU: params=11,181,642 train=0.33s train_loss=0.9546 inference=0.57ms/image test_loss=0.6413 accuracy=100.0%


MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1103: libMIOpenCKGroupedConv_gfx1103.so: cannot open shared object file: No such file or directory
MIOpen: Warning [ParseAndLoadDb] File is unreadable: "/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1103_6.HIP.fdb.txt"


ResNet18 iGPU: params=11,181,642 train=0.23s train_loss=0.9544 inference=0.65ms/image test_loss=0.6375 accuracy=100.0%
CPU vs iGPU (ResNet18 training time): CPU=0.326 s, iGPU=0.231 s; iGPU faster x1.41 (iGPU speedup x1.4110063)
CPU vs iGPU (ResNet18 inference latency): CPU=0.569 ms/image, iGPU=0.654 ms/image; CPU faster x1.15 (iGPU speedup x0.86906958)
MobileNetV3-Small CPU: params=1,528,106 train=0.67s train_loss=2.2647 inference=3.02ms/image test_loss=2.3028 accuracy=7.8%
MobileNetV3-Small iGPU: params=1,528,106 train=0.12s train_loss=2.2617 inference=0.19ms/image test_loss=2.3028 accuracy=7.8%
CPU vs iGPU (MobileNetV3-Small training time): CPU=0.673 s, iGPU=0.124 s; iGPU faster x5.41 (iGPU speedup x5.4088685)
CPU vs iGPU (MobileNetV3-Small inference latency): CPU=3.020 ms/image, iGPU=0.186 ms/image; iGPU faster x16.27 (iGPU speedup x16.267813)


0

## 4. Text generation

Runs `transformers_demo.py` on CPU and iGPU, reporting generated-token throughput and speedup. The default `distilgpt2` model is downloaded on first use.

In [5]:
run_benchmark("transformers_demo.py", "--device", DEVICE_MODE)


===== transformers_demo.py --device both =====
Prompt: Once upon a time
Input tokens: 4

Loading 'distilgpt2' on CPU (torch.float32)...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 17756.63it/s]


CPU generated text:  when there was a chance to have a conversation, I'd go back and ask myself, "What kind of conversation do you have with someone who's already had a conversation with you?" And then I'd say, "What are your feelings about you
CPU: tokens=50, time=6.997s, throughput=7.15 tokens/s

Loading 'distilgpt2' on iGPU (torch.float16)...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3034.14it/s]
/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/transformers/integrations/sdpa_attention.py:182: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:445.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/transformers/integrations/sdpa_attention.py:182: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:505.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


iGPU generated text: , you can see the sun in the sky, with a single glance.




When the sun is turned on, the sky will turn white.

The sun will turn black.
The sun will turn black.
The
iGPU: tokens=50, time=0.372s, throughput=134.39 tokens/s
Tokens generated: CPU=50, iGPU=50
CPU vs iGPU (text-generation throughput): CPU=7.146 tokens/s, iGPU=134.387 tokens/s; iGPU faster x18.81 (iGPU speedup x18.805444)


0

## 5. Sentence embeddings and semantic search

Runs `sentence_transformers_demo.py` on CPU and iGPU, reporting embedding throughput and semantic-search results. The default model is downloaded on first use.

In [6]:
run_benchmark("sentence_transformers_demo.py", "--device", DEVICE_MODE)


===== sentence_transformers_demo.py --device both =====
Query: How can I find related documents by meaning?

Loading 'sentence-transformers/all-MiniLM-L6-v2' on CPU...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16492.21it/s]


CPU: encoded 7 sentences in 1.871s (3.74 sentences/s); embedding size=384
Semantic search results:
1. score=0.5249  Semantic search finds passages by meaning rather than exact keywords.
2. score=0.1550  Sentence Transformers encode sentences as dense vector embeddings.
3. score=0.1475  Text generation models continue writing from a prompt.

Loading 'sentence-transformers/all-MiniLM-L6-v2' on iGPU...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15016.10it/s]
/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/transformers/integrations/sdpa_attention.py:182: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:445.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/transformers/integrations/sdpa_attention.py:182: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:505.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


iGPU: encoded 7 sentences in 0.201s (34.79 sentences/s); embedding size=384
Semantic search results:
1. score=0.5249  Semantic search finds passages by meaning rather than exact keywords.
2. score=0.1550  Sentence Transformers encode sentences as dense vector embeddings.
3. score=0.1475  Text generation models continue writing from a prompt.
CPU vs iGPU (sentence-embedding throughput): CPU=3.742 sentences/s, iGPU=34.787 sentences/s; iGPU faster x9.30 (iGPU speedup x9.2963493)


0

## 6. ResNet18 one-step probe

Runs the single-step FP32 CPU/iGPU comparison from `resnet18_probe.py`, including GPU memory and available kernel-fault diagnostics. **Even this small GPU workload could hang the system.**

In [7]:
run_benchmark("resnet18_probe.py", "--device", DEVICE_MODE)
print(f"Notebook output saved to {notebook_log}")
stop_notebook_logging()


===== resnet18_probe.py --device both =====
ResNet18 probe: fp32, one training step, batch=4, image=64x64
CPU training step completed: loss=2.2178, time=0.075s
iGPU: AMD Radeon 780M Graphics
Free GPU memory before test: 7.43/7.44 GiB


MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1103: libMIOpenCKGroupedConv_gfx1103.so: cannot open shared object file: No such file or directory
MIOpen: Warning [ParseAndLoadDb] File is unreadable: "/home/max/Projects/dl-rocm-demo/.venv/lib/python3.13/site-packages/_rocm_sdk_libraries/share/miopen/db/gfx1103_6.HIP.fdb.txt"


iGPU training step completed: loss=2.2178, time=0.537s
GPU free after step: 6.75/7.44 GiB; free-memory change: +0.68 GiB
PyTorch peak allocated/reserved: 0.28/0.29 GiB
GPU page-fault check: unavailable (dmesg: read kernel buffer failed: Operation not permitted)
CPU vs iGPU (ResNet18 training step): CPU=0.075 s, iGPU=0.537 s; CPU faster x7.15 (iGPU speedup x0.13985963)
Notebook output saved to /home/max/Projects/dl-rocm-demo/benchmark_results/demo_notebook.txt
